In [1]:
# Install pymysql, the package that lets Python talk to MySQL
# (only needed once per computer; safe to run again)

%pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Bring in the tools
# 1. pandas, for reading the clean CSVs
# 2. SQLAlchemy's create_engine and URL, for building the connection to MySQL
# 3. getpass, for typing the password without saving it in the notebook

import pandas as pd
from sqlalchemy import create_engine, URL
import getpass

In [3]:
# Connect to the project database
# 1. Ask for the MySQL password in a hidden box (so it's never saved or pushed to GitHub)
# 2. Build the connection details: user, password, server, port, database
# 3. Create the engine, Python's open line to MySQL

password = getpass.getpass("MySQL password: ")

connection_url = URL.create(
    "mysql+pymysql",
    username="root",
    password=password,
    host="localhost",
    port=3306,
    database="ev_charging_gap"
)

engine = create_engine(connection_url)

MySQL password:  ········


In [4]:
# Test the connection: ask MySQL to list the tables in the project database
# Expect 3: chargers_all_speeds, private_bevs, rapid_chargers

pd.read_sql("SHOW TABLES", engine)

,Tables_in_ev_charging_gap
0,chargers_all_speeds
1,europe_comparison
2,private_bevs
3,rapid_chargers


In [5]:
# Load the all-speed charger data into MySQL
# 1. Read the clean CSV from data/clean
# 2. Send it into the empty chargers_all_speeds table
#    - append: add rows to the existing table (keeps the column types we set in SQL)
#    - index=False: don't send pandas' row numbers as an extra column
# 3. Blank cells (NaN) become NULL in MySQL automatically

chargers = pd.read_csv("../data/clean/chargers_all_speeds.csv")

chargers.to_sql("chargers_all_speeds", engine, if_exists="append", index=False)

10780

In [6]:
# Check the load from MySQL's side: count the rows in the table
# Expect 10780

pd.read_sql("SELECT COUNT(*) FROM chargers_all_speeds", engine)

,COUNT(*)
0,10780


In [7]:
# Load the rapid charger and private BEV data into MySQL
# Same steps as for all-speed chargers: read the clean CSV, append it to its table
# Expect 4620 rapid charger rows and 24187 private BEV rows

rapid = pd.read_csv("../data/clean/rapid_chargers.csv")
rapid.to_sql("rapid_chargers", engine, if_exists="append", index=False)

bevs = pd.read_csv("../data/clean/private_bevs.csv")
bevs.to_sql("private_bevs", engine, if_exists="append", index=False)

24187

In [8]:
# Check all three tables from MySQL's side
# 1. Count the rows in each table
# 2. Stack the three answers into one result (UNION ALL)
# Expect 10780, 4620 and 24187

pd.read_sql("""
    SELECT 'chargers_all_speeds' AS table_name, COUNT(*) AS row_count FROM chargers_all_speeds
    UNION ALL
    SELECT 'rapid_chargers', COUNT(*) FROM rapid_chargers
    UNION ALL
    SELECT 'private_bevs', COUNT(*) FROM private_bevs
""", engine)

,table_name,row_count
0,chargers_all_speeds,10780
1,rapid_chargers,4620
2,private_bevs,24187


In [5]:
# Load the European comparison into MySQL
# 1. Read the clean CSV (IEA data, local only)
# 2. Append it to the empty europe_comparison table
# 3. Check from MySQL's side: expect 457 rows

europe = pd.read_csv("../data/clean/europe_comparison.csv")
europe.to_sql("europe_comparison", engine, if_exists="append", index=False)

pd.read_sql("SELECT COUNT(*) FROM europe_comparison", engine)

,COUNT(*)
0,457
